In [1]:
import pandas as pd
import numpy as np

# Datensatz laden
train_df = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW_NB15_training-set.csv")
test_df  = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW_NB15_testing-set.csv")

print("Shape Train:", train_df.shape)
print("Shape Test: ", test_df.shape)

# Alle Spaltennamen anzeigen
print("\nSpalten:\n", train_df.columns.tolist())

# Erste Zeilen
train_df.head()

Shape Train: (82332, 45)
Shape Test:  (175341, 45)

Spalten:
 ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [16]:
# 1. Was ist die 'id' Spalte?
print("id Spalte:")
print(train_df["id"].head(20).tolist())
print("Monoton steigend:", train_df["id"].is_monotonic_increasing)
print("Eindeutig:", train_df["id"].nunique() == len(train_df))

# 2. Label-Verteilung
print("\nLabel (0/1):")
print(train_df["label"].value_counts())

# 3. Angriffstypen
print("\nAngriffstypen (attack_cat):")
print(train_df["attack_cat"].value_counts())

# 4. Gibt es einen echten Timestamp versteckt?
print("\nAlle Spalten mit möglichem Zeitbezug:")
time_cols = [c for c in train_df.columns if any(
    x in c.lower() for x in ["time", "ts", "stamp", "date", "id"]
)]
print(time_cols)
for col in time_cols:
    print(f"\n{col} - erste 5 Werte: {train_df[col].head(5).tolist()}")

id Spalte:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Monoton steigend: True
Eindeutig: True

Label (0/1):
label
1    45332
0    37000
Name: count, dtype: int64

Angriffstypen (attack_cat):
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64

Alle Spalten mit möglichem Zeitbezug:
['id', 'spkts', 'dpkts', 'is_sm_ips_ports']

id - erste 5 Werte: [1, 2, 3, 4, 5]

spkts - erste 5 Werte: [2, 2, 2, 2, 2]

dpkts - erste 5 Werte: [0, 0, 0, 0, 0]

is_sm_ips_ports - erste 5 Werte: [0, 0, 0, 0, 0]


In [18]:
# Step 1: Load feature names from features file
features_df = pd.read_csv("../datasets/TONIoT_UNSW-NB15/NUSW-NB15_features.csv", encoding="latin-1")
print(features_df)

    No.              Name      Type   \
0     1             srcip    nominal   
1     2             sport    integer   
2     3             dstip    nominal   
3     4            dsport    integer   
4     5             proto    nominal   
5     6             state    nominal   
6     7               dur      Float   
7     8            sbytes    Integer   
8     9            dbytes    Integer   
9    10              sttl    Integer   
10   11              dttl    Integer   
11   12             sloss    Integer   
12   13             dloss    Integer   
13   14           service    nominal   
14   15             Sload      Float   
15   16             Dload      Float   
16   17             Spkts    integer   
17   18             Dpkts    integer   
18   19              swin    integer   
19   20              dwin    integer   
20   21             stcpb    integer   
21   22             dtcpb    integer   
22   23           smeansz    integer   
23   24           dmeansz    integer   


In [19]:
# Step 2: Extract column names from features file
# (check which column contains the feature names first)
print("Features file columns:", features_df.columns.tolist())
print(features_df.head(10))

Features file columns: ['No.', 'Name', 'Type ', 'Description']
   No.    Name    Type                                         Description
0    1   srcip  nominal                                  Source IP address
1    2   sport  integer                                 Source port number
2    3   dstip  nominal                             Destination IP address
3    4  dsport  integer                            Destination port number
4    5   proto  nominal                               Transaction protocol
5    6   state  nominal  Indicates to the state and its dependent proto...
6    7     dur    Float                              Record total duration
7    8  sbytes  Integer           Source to destination transaction bytes 
8    9  dbytes  Integer            Destination to source transaction bytes
9   10    sttl  Integer          Source to destination time to live value 


In [21]:
# Step 3: Load raw files with column names
import pandas as pd

# extract column name list from features file
col_names = features_df["Name"].tolist()  # adjust if column is named differently
print("Number of columns in features file:", len(col_names))

# load all 4 raw files
raw1 = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW-NB15_1.csv", header=None, names=col_names, low_memory=False)
raw2 = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW-NB15_2.csv", header=None, names=col_names, low_memory=False)
raw3 = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW-NB15_3.csv", header=None, names=col_names, low_memory=False)
raw4 = pd.read_csv("../datasets/TONIoT_UNSW-NB15/UNSW-NB15_4.csv", header=None, names=col_names, low_memory=False)

# combine all into one dataframe
raw_df = pd.concat([raw1, raw2, raw3, raw4], ignore_index=True)

print("Total shape:", raw_df.shape)
print("\nColumns:", raw_df.columns.tolist())
print("\nFirst row:")
print(raw_df.iloc[0])

Number of columns in features file: 49
Total shape: (2540047, 49)

Columns: ['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']

First row:
srcip                  59.166.0.0
sport                        1390
dstip               149.171.126.6
dsport                         53
proto                         udp
state                         CON
dur                      0.001055
sbytes                        132
dbytes                        164
sttl                           31
dttl         

In [22]:
# Step 4: Check for timestamps and IP addresses
print("First 5 values of each potentially relevant column:")
for col in raw_df.columns:
    if any(x in col.lower() for x in ["time", "ip", "addr", "src", "dst", "ts"]):
        print(f"\n{col}: {raw_df[col].head(5).tolist()}")

First 5 values of each potentially relevant column:

srcip: ['59.166.0.0', '59.166.0.0', '59.166.0.6', '59.166.0.5', '59.166.0.3']

dstip: ['149.171.126.6', '149.171.126.9', '149.171.126.7', '149.171.126.5', '149.171.126.0']

Spkts: [2, 4, 2, 2, 2]

Dpkts: [2, 4, 2, 2, 2]

Stime: [1421927414, 1421927414, 1421927414, 1421927414, 1421927414]

Ltime: [1421927414, 1421927414, 1421927414, 1421927414, 1421927414]

is_sm_ips_ports: [0, 0, 0, 0, 0]

ct_srv_src: [3, 2, 12, 6, 7]

ct_srv_dst: [7, 4, 8, 9, 9]

ct_dst_ltm: [1, 2, 1, 1, 1]

ct_src_ ltm: [3, 3, 2, 1, 1]

ct_src_dport_ltm: [1, 1, 2, 1, 1]

ct_dst_sport_ltm: [1, 1, 1, 1, 1]

ct_dst_src_ltm: [1, 2, 1, 1, 1]


In [23]:
# label distribution in raw data
print("Label distribution (0=Benign, 1=Attack):")
print(raw_df["Label"].value_counts())
print("\nPercentage:")
print(raw_df["Label"].value_counts(normalize=True).map("{:.2%}".format))

# attack category distribution
print("\nAttack categories:")
print(raw_df["attack_cat"].fillna("Normal").value_counts())
print("\nPercentage:")
print(raw_df["attack_cat"].fillna("Normal").value_counts(normalize=True).map("{:.2%}".format))

# temporal distribution: benign vs attack over time
raw_df["Stime_converted"] = pd.to_datetime(raw_df["Stime"], unit="s")

print("\nTime range:")
print(f"  Start: {raw_df['Stime_converted'].min()}")
print(f"  End:   {raw_df['Stime_converted'].max()}")
print(f"  Total duration: {raw_df['Stime_converted'].max() - raw_df['Stime_converted'].min()}")

# benign vs attack time range
for label, name in [(0, "Benign"), (1, "Attack")]:
    subset = raw_df[raw_df["Label"] == label]["Stime_converted"]
    print(f"\n{name}: {subset.min()}  →  {subset.max()}")

Label distribution (0=Benign, 1=Attack):
Label
0    2218764
1     321283
Name: count, dtype: int64

Percentage:
Label
0    87.35%
1    12.65%
Name: proportion, dtype: str

Attack categories:
attack_cat
Normal              2218764
Generic              215481
Exploits              44525
 Fuzzers              19195
DoS                   16353
 Reconnaissance       12228
 Fuzzers               5051
Analysis               2677
Backdoor               1795
Reconnaissance         1759
 Shellcode             1288
Backdoors               534
Shellcode               223
Worms                   174
Name: count, dtype: int64

Percentage:
attack_cat
Normal              87.35%
Generic              8.48%
Exploits             1.75%
 Fuzzers             0.76%
DoS                  0.64%
 Reconnaissance      0.48%
 Fuzzers             0.20%
Analysis             0.11%
Backdoor             0.07%
Reconnaissance       0.07%
 Shellcode           0.05%
Backdoors            0.02%
Shellcode            0.01%
Worms

In [24]:
# How many unique source IPs (= nodes)?
print("Unique srcip (nodes):", raw_df["srcip"].nunique())
print("Top 10 source IPs:")
print(raw_df["srcip"].value_counts().head(10))

# Does each node have both benign and attack traffic?
node_label = raw_df.groupby("srcip")["Label"].agg(
    benign=lambda x: (x == 0).sum(),
    attack=lambda x: (x == 1).sum(),
    total="count"
)
print("\nNodes with both benign AND attack traffic:")
mixed_nodes = node_label[(node_label["benign"] > 0) & (node_label["attack"] > 0)]
print(f"  {len(mixed_nodes)} of {len(node_label)} nodes")
print(mixed_nodes.head(10))

Unique srcip (nodes): 43
Top 10 source IPs:
srcip
59.166.0.4    197959
59.166.0.1    197680
59.166.0.5    197626
59.166.0.2    197550
59.166.0.0    197528
59.166.0.3    195953
59.166.0.9    190187
59.166.0.6    189419
59.166.0.8    189341
59.166.0.7    189059
Name: count, dtype: int64

Nodes with both benign AND attack traffic:
  4 of 43 nodes
              benign  attack   total
srcip                               
175.45.176.0    8720   74279   82999
175.45.176.1   10090  117908  127998
175.45.176.2    9348   22678   32026
175.45.176.3   12255  106418  118673


In [25]:
# Sort by time
raw_df_sorted = raw_df.sort_values("Stime").reset_index(drop=True)

# Create time windows (e.g. 60 seconds)
window_seconds = 60
raw_df_sorted["time_window"] = (
    (raw_df_sorted["Stime"] - raw_df_sorted["Stime"].min()) // window_seconds
).astype(int)

# Ground-Truth per node per time window
ground_truth = raw_df_sorted.groupby(["srcip", "time_window"])["Label"].agg(
    benign=lambda x: (x == 0).sum(),
    attack=lambda x: (x == 1).sum(),
    label_majority=lambda x: int(x.mean() >= 0.5)
).reset_index()

print("Ground-Truth Tabelle (Knoten x Zeitfenster):")
print(ground_truth.shape)
print(ground_truth.head(20))

Ground-Truth Tabelle (Knoten x Zeitfenster):
(27213, 5)
          srcip  time_window  benign  attack  label_majority
0   10.40.170.2            0       2       0               0
1   10.40.170.2            1       2       0               0
2   10.40.170.2            2       2       0               0
3   10.40.170.2            3       2       0               0
4   10.40.170.2            4       2       0               0
5   10.40.170.2            5       2       0               0
6   10.40.170.2            6       2       0               0
7   10.40.170.2            7       2       0               0
8   10.40.170.2            8       2       0               0
9   10.40.170.2            9       2       0               0
10  10.40.170.2           10       2       0               0
11  10.40.170.2           11       2       0               0
12  10.40.170.2           12       2       0               0
13  10.40.170.2           13       2       0               0
14  10.40.170.2           14 

In [26]:
# Are there time windows where the same node has MIXED traffic?
# (both benign and attack in same window = local misdetection scenario)
mixed_windows = ground_truth[
    (ground_truth["benign"] > 0) & (ground_truth["attack"] > 0)
]
print(f"Timewindow with mixed Traffic (Benign+Attack): {len(mixed_windows)}")
print(f"Anteil: {len(mixed_windows)/len(ground_truth):.2%}")
print(mixed_windows.head(10))

Timewindow with mixed Traffic (Benign+Attack): 1399
Anteil: 5.14%
             srcip  time_window  benign  attack  label_majority
9688  175.45.176.0            5       7      41               1
9690  175.45.176.0            7      13      67               1
9691  175.45.176.0            8      16      18               1
9692  175.45.176.0            9       1      29               1
9695  175.45.176.0           12       8      21               1
9700  175.45.176.0           17      26      35               1
9701  175.45.176.0           18      32      23               0
9702  175.45.176.0           19      86      35               0
9703  175.45.176.0           20      70      80               1
9704  175.45.176.0           21      23      20               0


In [27]:
# filter to only the 4 relevant attack nodes
attack_nodes = ["175.45.176.0", "175.45.176.1", "175.45.176.2", "175.45.176.3"]
focused_df = raw_df_sorted[raw_df_sorted["srcip"].isin(attack_nodes)].copy()

print("Shape focused dataset:", focused_df.shape)
print("\nLabel distribution:")
print(focused_df["Label"].value_counts())
print(focused_df["Label"].value_counts(normalize=True).map("{:.2%}".format))

Shape focused dataset: (361696, 51)

Label distribution:
Label
1    321283
0     40413
Name: count, dtype: int64
Label
1    88.83%
0    11.17%
Name: proportion, dtype: str


In [28]:
# ============================================================
# DATASET EVALUATION SUMMARY – UNSW-NB15 (Raw Files 1–4)
# ============================================================

summary = """
DATASET: UNSW-NB15 (Raw Data, combined from UNSW-NB15_1.csv to UNSW-NB15_4.csv)
Total Records: 2,540,047 flows | 49 features

CRITERIA CHECK:
──────────────────────────────────────────────────────────────
[✅] Timestamps / Sliding Window
     - Column 'Stime' contains Unix timestamps
     - Time range: 2015-01-22 to 2015-02-18 (27 days)
     - Sliding windows directly constructable via Stime

[✅] Binary Labels (0/1)
     - Label 0 = Benign (2,218,764 records = 87.35%)
     - Label 1 = Attack  (321,283 records  = 12.65%)

[✅] Benign Phases present
     - 87.35% benign traffic enables FP baseline comparison
     - Benign and attack traffic overlap temporally

[✅] Multiple Attack Types
     - Generic, Exploits, DoS, Fuzzers, Reconnaissance,
       Backdoor, Shellcode, Analysis, Worms

[✅] Network-level Features
     - 49 features: proto, sbytes, dbytes, sttl, rate,
       Sload, Dload, tcprtt, service, state, ...

[⚠️] Multiple Simulated Nodes
     - 43 unique source IPs (srcip) identified as nodes
     - Only 4 nodes carry attack traffic:
         175.45.176.0 –  8,720 benign |  74,279 attack
         175.45.176.1 – 10,090 benign | 117,908 attack
         175.45.176.2 –  9,348 benign |  22,678 attack
         175.45.176.3 – 12,255 benign | 106,418 attack
     - Remaining 39 nodes: benign-only (useful as context)

[✅] Ground-Truth Labels per Node & Time Window
     - 27,213 unique (node, time_window) combinations
     - Window size: 60 seconds
     - Enables per-node TP, FP, TN, FN computation

[✅] Local Misdetection Scenarios
     - 1,399 time windows contain mixed benign+attack traffic
     - Represents 5.14% of all node-window combinations
     - Enables demonstration of local detection errors

──────────────────────────────────────────────────────────────
METRICS FEASIBILITY:
     - Recall (TPR):  computable per node and time window
     - FPR:           computable, strong benign baseline available
     - F1-Score:      directly computable from TP/FP/TN/FN

CONCLUSION:
     The UNSW-NB15 raw dataset satisfies all core requirements
     for evaluating a distributed Detection and Reaction System.
     The 4 attack nodes serve as local detectors, while the 39
     benign-only nodes provide aggregation context to reduce
     false positives and improve overall detection performance.
──────────────────────────────────────────────────────────────
"""

print(summary)


DATASET: UNSW-NB15 (Raw Data, combined from UNSW-NB15_1.csv to UNSW-NB15_4.csv)
Total Records: 2,540,047 flows | 49 features

CRITERIA CHECK:
──────────────────────────────────────────────────────────────
[✅] Timestamps / Sliding Window
     - Column 'Stime' contains Unix timestamps
     - Time range: 2015-01-22 to 2015-02-18 (27 days)
     - Sliding windows directly constructable via Stime

[✅] Binary Labels (0/1)
     - Label 0 = Benign (2,218,764 records = 87.35%)
     - Label 1 = Attack  (321,283 records  = 12.65%)

[✅] Benign Phases present
     - 87.35% benign traffic enables FP baseline comparison
     - Benign and attack traffic overlap temporally

[✅] Multiple Attack Types
     - Generic, Exploits, DoS, Fuzzers, Reconnaissance,
       Backdoor, Shellcode, Analysis, Worms

[✅] Network-level Features
     - 49 features: proto, sbytes, dbytes, sttl, rate,
       Sload, Dload, tcprtt, service, state, ...

[⚠️] Multiple Simulated Nodes
     - 43 unique source IPs (srcip) identifie

In [29]:
# Verify: how many rows per node?
print(raw_df_sorted.groupby("srcip").size().head(10))

# Verify: time difference between consecutive rows per node
raw_df_sorted["time_diff"] = raw_df_sorted.groupby("srcip")["Stime"].diff()
print("\nTime difference between rows (should be ~30s):")
print(raw_df_sorted["time_diff"].describe())
print("\nMost common time differences:")
print(raw_df_sorted["time_diff"].value_counts().head(5))

srcip
10.40.170.2      2094
10.40.182.1      3984
10.40.182.3      2105
10.40.182.6      3492
10.40.85.1       4018
10.40.85.10       793
10.40.85.30      2138
127.0.0.1           1
149.171.126.0     261
149.171.126.1     311
dtype: int64

Time difference between rows (should be ~30s):
count    2.540004e+06
mean     3.492471e+01
std      8.758170e+03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.301057e+06
Name: time_diff, dtype: float64

Most common time differences:
time_diff
0.0    1862390
1.0     441011
2.0     137518
3.0      49307
4.0      19000
Name: count, dtype: int64


In [30]:
# use the variables exactly as defined in your notebook:
# attack_nodes, focused_df

# 1) Which columns have at least one non-null value for every one of the 4 nodes?
features_present_in_all_4 = []

for col in focused_df.columns:
    has_data_for_all = all(
        not focused_df.loc[focused_df["srcip"] == node, col].dropna().empty
        for node in attack_nodes
    )
    if has_data_for_all:
        features_present_in_all_4.append(col)

print("Features present for all 4 attack nodes:")
print(features_present_in_all_4)
print("\nCount:", len(features_present_in_all_4))

Features present for all 4 attack nodes:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label', 'Stime_converted', 'time_window']

Count: 51


In [31]:
# ============================================================
# FEATURE OVERVIEW – UNSW-NB15 (Interpretation for Notes)
# ============================================================

feature_summary = """
DATASET: UNSW-NB15 (Raw Features for Flow-based Attack Detection)
Feature Basis: 49 raw features + derived columns in notebook
Node Definition in this analysis: srcip as local node identifier

FEATURE GROUPS:
──────────────────────────────────────────────────────────────
[1] Communication / Endpoint Features
     - srcip:   source IP address
     - dstip:   destination IP address
     - sport:   source port
     - dsport:  destination port
     - These features describe which node communicates with
       which target and through which ports.

[2] Protocol / Session Type Features
     - proto:   transaction protocol
     - state:   connection state and protocol-dependent state
     - service: application/service information
     - These features characterize the type of communication
       and the session behavior of a flow.

[3] Traffic Volume / Load Features
     - dur:     total duration of the record
     - sbytes:  bytes sent from source to destination
     - dbytes:  bytes sent from destination to source
     - Spkts:   packets from source to destination
     - Dpkts:   packets from destination to source
     - Sload:   source bits per second
     - Dload:   destination bits per second
     - These features describe how much traffic a flow causes
       in terms of duration, size, packet count, and rate.

[4] Timing / Temporal Behavior Features
     - Stime:   record start time
     - Ltime:   record last time
     - Sintpkt: source inter-packet arrival time
     - Dintpkt: destination inter-packet arrival time
     - Sjit:    source jitter
     - Djit:    destination jitter
     - These features capture temporal patterns and timing
       behavior of the communication.

[5] TCP Setup / Connection State Features
     - tcprtt:  TCP setup round-trip time
     - synack:  time between SYN and SYN_ACK
     - ackdat:  time between SYN_ACK and ACK
     - swin:    source TCP window advertisement value
     - dwin:    destination TCP window advertisement value
     - stcpb:   source TCP base sequence number
     - dtcpb:   destination TCP base sequence number
     - These features describe properties of TCP connection
       establishment and low-level connection state behavior.

[6] Flow Context / Aggregation Features
     - ct_srv_src:         same service and source address
     - ct_srv_dst:         same service and destination address
     - ct_dst_ltm:         same destination address in time context
     - ct_src_dport_ltm:   same source and destination port
     - ct_dst_sport_ltm:   same destination and source port
     - ct_dst_src_ltm:     same source-destination pair
     - ct_state_ttl:       state/ttl related counting feature
     - ct_flw_http_mthd:   number of flows with HTTP methods
     - These features capture recurring patterns across
       multiple similar flows and provide behavioral context
       beyond a single packet or single connection.

[7] Additional Behavioral Features
     - sttl, dttl:         source/destination time-to-live
     - sloss, dloss:       retransmitted or dropped packets
     - smeansz, dmeansz:   mean packet sizes
     - trans_depth:        pipelined depth into the connection
     - res_bdy_len:        response body length
     - is_sm_ips_ports:    binary relation of source/destination IPs and ports
     - is_ftp_login:       ftp login indicator
     - ct_ftp_cmd:         number of ftp commands
     - These features provide additional evidence about
       packet structure, retransmission behavior, protocol
       specifics, and application-level behavior.

[8] Target Variables
     - attack_cat: attack category
     - Label:      binary label (0 = normal, 1 = attack)
     - These are not detector input features in the strict
       sense, but ground-truth variables for supervised
       learning and evaluation.

──────────────────────────────────────────────────────────────
DETECTION RELEVANCE:
     - The feature set combines endpoint information, protocol
       semantics, traffic volume, temporal behavior, TCP setup
       characteristics, and multi-flow context.
     - Therefore, the dataset provides enough observable
       network behavior to distinguish benign from malicious
       traffic in principle.
     - The notebook analysis already confirms that binary
       labels, attack categories, timestamps, sliding windows,
       and per-node / per-window ground truth are available.

THEORETICAL INTERPRETATION:
     - Attacks can theoretically be recognized when feature
       values or feature combinations deviate from benign
       communication patterns.
     - Possible indicators are unusual traffic rates, uncommon
       protocol-state combinations, abnormal timing behavior,
       repeated similar flows, or suspicious TCP setup values.
     - In this dataset, detection is not based on one single
       feature alone, but on the joint behavior of multiple
       features over flows, nodes, and time windows.

IMPORTANT NOTE FOR PAPER WRITING:
     - Strictly speaking, the features belong to flows/records,
       not to nodes directly.
     - In this notebook, nodes are represented by srcip.
     - A precise formulation is therefore:
       "The analysis uses flow-level features and aggregates
       them per srcip-defined node and time window."

──────────────────────────────────────────────────────────────
CONCLUSION:
     The UNSW-NB15 feature space is well suited for flow-based
     and node-aware attack detection because it contains both
     low-level traffic measurements and higher-level context
     features, together with binary and categorical ground truth.
──────────────────────────────────────────────────────────────
"""

print(feature_summary)


DATASET: UNSW-NB15 (Raw Features for Flow-based Attack Detection)
Feature Basis: 49 raw features + derived columns in notebook
Node Definition in this analysis: srcip as local node identifier

FEATURE GROUPS:
──────────────────────────────────────────────────────────────
[1] Communication / Endpoint Features
     - srcip:   source IP address
     - dstip:   destination IP address
     - sport:   source port
     - dsport:  destination port
     - These features describe which node communicates with
       which target and through which ports.

[2] Protocol / Session Type Features
     - proto:   transaction protocol
     - state:   connection state and protocol-dependent state
     - service: application/service information
     - These features characterize the type of communication
       and the session behavior of a flow.

[3] Traffic Volume / Load Features
     - dur:     total duration of the record
     - sbytes:  bytes sent from source to destination
     - dbytes:  bytes sent

In [32]:
# ============================================================
# NODE-LOCAL ML MODEL STRUCTURE
# Prepare dataset so each node can run its own model
# ============================================================

# Step 1: Identify all nodes and their properties
nodes_info = raw_df_sorted.groupby("srcip").agg({
    "Label": ["count", lambda x: (x == 0).sum(), lambda x: (x == 1).sum()],
    "Stime": ["min", "max"],
    "dstip": "nunique",
    "sport": "nunique",
    "dsport": "nunique"
}).round(2)

nodes_info.columns = ["total_flows", "benign_flows", "attack_flows", 
                       "time_start", "time_end", "unique_targets", "unique_sports", "unique_dports"]
nodes_info["attack_ratio"] = (nodes_info["attack_flows"] / nodes_info["total_flows"] * 100).round(2)

print("=" * 80)
print("NODE OVERVIEW – Each node will run its own model")
print("=" * 80)
print(nodes_info.sort_values("total_flows", ascending=False))

print(f"\n✓ Total nodes: {len(nodes_info)}")
print(f"✓ Nodes with attack traffic: {(nodes_info['attack_flows'] > 0).sum()}")
print(f"✓ Nodes with benign-only traffic: {(nodes_info['attack_flows'] == 0).sum()}")


NODE OVERVIEW – Each node will run its own model
                 total_flows  benign_flows  attack_flows  time_start  \
srcip                                                                  
59.166.0.4            197959        197959             0  1421927415   
59.166.0.1            197680        197680             0  1421927415   
59.166.0.5            197626        197626             0  1421927414   
59.166.0.2            197550        197550             0  1421927415   
59.166.0.0            197528        197528             0  1421927414   
59.166.0.3            195953        195953             0  1421927414   
59.166.0.9            190187        190187             0  1421927415   
59.166.0.6            189419        189419             0  1421927414   
59.166.0.8            189341        189341             0  1421927416   
59.166.0.7            189059        189059             0  1421927415   
175.45.176.1          127998         10090        117908  1421927425   
175.45.176.3   

In [33]:
# Check that Stime can be used for temporally aligned node comparison

print("Stime min:", raw_df["Stime"].min())
print("Stime max:", raw_df["Stime"].max())

raw_df["Stime_converted"] = pd.to_datetime(raw_df["Stime"], unit="s")
print("Converted start:", raw_df["Stime_converted"].min())
print("Converted end:  ", raw_df["Stime_converted"].max())

window_seconds = 60
raw_df["time_window"] = ((raw_df["Stime"] - raw_df["Stime"].min()) // window_seconds).astype(int)

node_window_counts = raw_df.groupby("time_window")["srcip"].nunique()

print("\nTime windows total:", raw_df["time_window"].nunique())
print("Windows with more than one node:", (node_window_counts > 1).sum())
print("Max nodes in one window:", node_window_counts.max())

print("\nExample node counts per time window:")
print(node_window_counts.head(10))

Stime min: 1421927377
Stime max: 1424262068
Converted start: 2015-01-22 11:49:37
Converted end:   2015-02-18 12:21:08

Time windows total: 1441
Windows with more than one node: 1441
Max nodes in one window: 29

Example node counts per time window:
time_window
0    20
1    20
2    20
3    20
4    20
5    20
6    20
7    20
8    20
9    19
Name: srcip, dtype: int64
